In [29]:
import tensorflow as tf
import pathlib
import gzip
import shutil
from scipy.signal import resample
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models
import tensorflow_hub as hub
import pandas as pd
import io
import librosa
import os

In [7]:
gz_path = 'dataset_commands-002.gz'

In [16]:
def le_arquivos(gz_path):
    extracted_path = 'tmp'
    
    with gzip.open(gz_path, 'rb') as f_in:
        with open(extracted_path + '.tar', 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
            
    # Extrair o arquivo .tar resultante
    shutil.unpack_archive(extracted_path + '.tar', extracted_path)
    
    data_dir = pathlib.Path(extracted_path)
    
    all_audio_paths = list(data_dir.glob('*/**/*.wav'))
    all_labels = [path.parent.name for path in all_audio_paths]
    
    # Converter caminhos para strings
    all_audio_paths = [str(path) for path in all_audio_paths]
    
    return all_audio_paths, all_labels

In [18]:
all_audio_paths, all_labels = le_arquivos(gz_path)

OSError: [Errno 28] No space left on device

In [ ]:
np.unique(all_labels)

In [ ]:
np.unique(all_labels).shape

In [19]:
example_audio_path = all_audio_paths[0]

NameError: name 'all_audio_paths' is not defined

In [ ]:
# Carregar o arquivo de áudio
audio_binary = tf.io.read_file(example_audio_path)
audio, _ = tf.audio.decode_wav(audio_binary)
audio = tf.squeeze(audio, axis=-1)

In [ ]:
# Plotar a forma de onda
plt.figure(figsize=(10, 6))
plt.plot(audio.numpy())
plt.title(f'Forma de onda para {example_audio_path}')
plt.xlabel('Amostras')
plt.ylabel('Amplitude')
plt.show()

## Processando os dados de áudio

In [ ]:
def load_and_process_audio(filename, max_length=16000):
    file_contents = tf.io.read_file(filename)
    wav, sample_rate = tf.audio.decode_wav(file_contents, desired_channels=1)
    wav = tf.squeeze(wav, axis=-1)
    
    # Função de resampling usando SciPy
    def scipy_resample(wav, sample_rate):
        if sample_rate != 16000:
            wav = resample(wav, int(16000 / sample_rate * len(wav)))
        return wav

    # Usar tf.py_function para envolver a operação de resampling
    wav = tf.py_function(scipy_resample, [wav, sample_rate], tf.float32)
    
    # Adicionar padding ou cortar os sinais de áudio
    audio_length = tf.shape(wav)[0]
    if audio_length > max_length:
        wav = wav[:max_length]
    else:
        pad_length = max_length - audio_length
        paddings = [[0, pad_length]]
        wav = tf.pad(wav, paddings, "CONSTANT")
    
    return tf.reshape(wav, [max_length])

def process_path(file_path, label):
    audio = load_and_process_audio(file_path)
    return audio, label

def paths_and_labels_to_dataset(audio_paths, labels):
    path_ds = tf.data.Dataset.from_tensor_slices(audio_paths)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    audio_label_ds = tf.data.Dataset.zip((path_ds, label_ds))
    return audio_label_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

def prepare_for_training(ds, batch_size=32, shuffle_buffer_size=1000):
    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

# Codificar as labels como inteiros
label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)

In [ ]:
complete_dataset = paths_and_labels_to_dataset(all_audio_paths, all_labels_encoded)

## Treinando a rede

In [ ]:
total_size = len(all_audio_paths)
val_size = int(0.02 * total_size)
train_size = total_size - val_size

complete_dataset = complete_dataset.shuffle(buffer_size=total_size, seed=42)
train_dataset = complete_dataset.take(train_size)
val_dataset = complete_dataset.skip(train_size)

In [ ]:
train_dataset = prepare_for_training(train_dataset)
val_dataset = prepare_for_training(val_dataset)

In [ ]:
model_time_domain = models.Sequential([
    layers.Input(shape=(16000, 1)),
    layers.Conv1D(16, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(36, activation='softmax')
])

model_time_domain.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])

history_time_domain = model_time_domain.fit(train_dataset, epochs=10, validation_data=val_dataset)

In [ ]:
def plot_history(history):
    # Resumo do histórico de precisão
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Acurácia de Treinamento')
    plt.plot(history.history['val_accuracy'], label='Acurácia de Validação')
    plt.title('Acurácia do Modelo')
    plt.xlabel('Época')
    plt.ylabel('Acurácia')
    plt.legend(loc='lower right')

    # Resumo do histórico de perda
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Perda de Treinamento')
    plt.plot(history.history['val_loss'], label='Perda de Validação')
    plt.title('Perda do Modelo')
    plt.xlabel('Época')
    plt.ylabel('Perda')
    plt.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

plot_history(history_time_domain)

## Mudando o domínio e aplicando FFT

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(audio.numpy())
plt.title(f'Forma de onda para {example_audio_path}')
plt.xlabel('Amostras')
plt.ylabel('Amplitude')
plt.show()

In [ ]:

def espectrograma(formadeonda):
    # Converte a forma de onda para um espectro grama usando STFT.
    espect = tf.signal.stft(
      formadeonda, frame_length=255, frame_step=128)

    espect = tf.abs(espect)

    # Adiciona uma dimensão `channels`, 
    # para que o espectrograma possa ser usado 
    # como dados de entrada semelhantes a imagens 
    # com camadas de convolução 
    # (que esperam # formato (`batch_size`, `height`, `width`, `channels`).
    espect = espect[..., tf.newaxis]
    return espect

espect = espectrograma(audio.numpy())

In [ ]:
def plota_espectrograma(espectrograma):
    if len(espectrograma.shape) > 2:
        assert len(espectrograma.shape) == 3
        espectrograma = np.squeeze(espectrograma, axis=-1)
    # Convert the frequencies to log scale and transpose, so that the time is
    # represented on the x-axis (columns).
    # Add an epsilon to avoid taking a log of zero.
    log_spec = np.log(espectrograma.T + np.finfo(float).eps)
    height = log_spec.shape[0]
    width = log_spec.shape[1]
    X = np.linspace(0, np.size(espectrograma), num=width, dtype=int)
    Y = range(height)
    plt.pcolormesh(X, Y, log_spec)

plota_espectrograma(espect)

In [ ]:
def get_spectrogram_and_label_id(audio, label):
    espect = espectrograma(audio)
    return espect, label

train_spec = train_dataset.map(map_func=get_spectrogram_and_label_id,num_parallel_calls=tf.data.AUTOTUNE)
val_spec = val_dataset.map(map_func=get_spectrogram_and_label_id,num_parallel_calls=tf.data.AUTOTUNE)
     

In [ ]:
# Normalize the spectrograms.
norm_layer = tf.keras.layers.Normalization()
# Get a batch of spectrograms to adapt the norm layer
for spectrogram, _ in train_spec.take(1):
    norm_layer.adapt(spectrogram)
     

In [ ]:
# Number of labels
num_labels = len(np.unique(all_labels))

In [ ]:
# Get the input shape from the spectrograms
for spectrogram, _ in train_spec.take(1):
    input_shape = spectrogram.shape[1:]

## Adaptando a rede

In [ ]:
input_shape

In [ ]:
train_spec = train_dataset.map(map_func=get_spectrogram_and_label_id, num_parallel_calls=tf.data.AUTOTUNE)
val_spec = val_dataset.map(map_func=get_spectrogram_and_label_id, num_parallel_calls=tf.data.AUTOTUNE)
     

In [ ]:
model_spectrogram = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Resizing(32, 32),
    norm_layer,
    layers.Conv2D(32, 3, activation='relu'),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_labels, activation='softmax')
])

model_spectrogram.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])
     

history_spectrogram = model_spectrogram.fit(train_spec, epochs=10, validation_data=val_spec)
     

In [ ]:
plot_history(history_spectrogram)

## Implementando uma camada de atenção de canal

Funcionamento da Camada de Atenção de Canal A camada de atenção de canal funciona da seguinte maneira:

Pooling Global: São realizadas operações de GlobalAveragePooling2D e GlobalMaxPooling2D para obter a média e o máximo globais de cada canal, resultando em dois vetores de características representando a importância média e máxima de cada canal.

Camadas Densas: Os vetores de características são passados por duas camadas densas. A primeira camada reduz a dimensionalidade (controlada pelo parâmetro ratio), enquanto a segunda camada retorna a atenção (pesos) para cada canal.

Combinação e Aplicação de Atenção: Os vetores de atenção resultantes das operações de pooling global são combinados e aplicados aos canais de entrada originais. Isso ajusta os valores dos canais com base em sua importância calculada.

In [ ]:
@tf.keras.utils.register_keras_serializable()
class ChannelAttention(tf.keras.layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.ratio = ratio
        self.avg_pool = layers.GlobalAveragePooling2D()
        self.max_pool = layers.GlobalMaxPooling2D()

    def build(self, input_shape):
        self.fc1 = layers.Dense(units=input_shape[-1] // self.ratio, activation='relu')
        self.fc2 = layers.Dense(units=input_shape[-1], activation='sigmoid')

    def call(self, inputs):
        avg_out = self.avg_pool(inputs)
        max_out = self.max_pool(inputs)
        avg_out = self.fc2(self.fc1(avg_out))
        max_out = self.fc2(self.fc1(max_out))
        out = avg_out + max_out
        out = tf.expand_dims(tf.expand_dims(out, axis=1), axis=1)
        return inputs * out


In [ ]:
model_spectrogram = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Resizing(32, 32),
    norm_layer,
    layers.Conv2D(32, 3, activation='relu'),
    ChannelAttention(ratio=8),
    layers.Conv2D(64, 3, activation='relu'),
    ChannelAttention(ratio=8),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_labels, activation='softmax')
])

model_spectrogram.compile(optimizer='adam', 
                          loss='sparse_categorical_crossentropy', 
                          metrics=['accuracy'])

history_spectrogram = model_spectrogram.fit(train_spec, epochs=10, validation_data=val_spec)

In [ ]:
plot_history(history_spectrogram)

## Salvando e fazendo uma inferência

In [ ]:
specific_audio_path = '/teamspace/studios/this_studio/yes.wav'

In [ ]:
specific_audio = load_and_process_audio(specific_audio_path)
specific_spectrogram = espectrograma(specific_audio.numpy())

In [ ]:
# Expandir a dimensão do batch para compatibilidade com o modelo
specific_spectrogram_batch = np.expand_dims(specific_spectrogram, axis=0)

# Fazer a predição
specific_predictions = model_spectrogram.predict(specific_spectrogram_batch)
specific_predicted_label = np.argmax(specific_predictions, axis=-1)[0]

In [ ]:
predicted_class_name = label_encoder.inverse_transform([specific_predicted_label])[0]
predicted_class_name

In [ ]:
model_spectrogram.save("model_spectrogram.keras")

## Trabalhando com modelos pré-treinados como Yamnet

In [26]:
#!pip install tensorflow_hub

In [ ]:
model = hub.load('https://www.kaggle.com/models/google/yamnet/TensorFlow2/yamnet/1')

waveform = np.zeros(3 * 16000, dtype=np.float32)
scores, embeddings, log_mel_spectrogram = model(waveform)
scores

In [ ]:
def class_names_from_csv(class_map_csv_text):
    """Retorna uma lista de nomes de classes correspondentes ao vetor de pontuação."""
    class_map_df = pd.read_csv(io.StringIO(class_map_csv_text))
    class_names = class_map_df['display_name'].tolist()
    return class_names  

class_map_path = model.class_map_path().numpy()

class_names = class_names_from_csv(tf.io.read_file(class_map_path).numpy().decode('utf-8'))
class_names

In [ ]:
class_names[np.argmax(scores.numpy().mean(axis=0))]

## Aplicando em diferentes tipos de áudio

In [ ]:
def load_audio(file_path):
    """
    Carrega um arquivo de áudio MP3 e retorna o waveform e a taxa de amostragem.

    Args:
    file_path (str): Caminho para o arquivo de áudio MP3.

    Returns:
    tuple: waveform (np.ndarray), sample_rate (int)
    """
    waveform, sample_rate = librosa.load(file_path, sr=None)
    return waveform, sample_rate

In [ ]:
file_path = '/teamspace/studios/this_studio/dog-barking-70772.mp3'

waveform, sample_rate = load_audio(file_path)
plt.plot(waveform)

In [ ]:
scores, embeddings, log_mel_spectrogram = model(waveform)
class_names[np.argmax(scores.numpy().mean(axis=0))]

In [ ]:
scores

In [ ]:
def plot_top_classes(scores, class_names, top_n=10):
    """
    Plota as top N classes mais prováveis com base nos scores.

    Args:
    scores (np.ndarray): Vetor de pontuações.
    class_names (list): Lista de nomes das classes.
    top_n (int): Número de top classes a serem visualizadas.
    """
    mean_scores = scores.mean(axis=0)
    top_indices = np.argsort(mean_scores)[-top_n:][::-1]
    top_scores = mean_scores[top_indices]
    top_class_names = [class_names[i] for i in top_indices]

    plt.figure(figsize=(10, 6))
    plt.barh(top_class_names, top_scores)
    plt.xlabel('Score')
    plt.ylabel('Class')
    plt.title(f'Top {top_n} Classes')
    plt.gca().invert_yaxis()
    plt.show()

plot_top_classes(scores.numpy(), class_names)

## Obtendo um novo dataset

In [ ]:
_ = tf.keras.utils.get_file('esc-50.zip',
                        'https://github.com/karoldvl/ESC-50/archive/master.zip',
                        cache_dir='./',
                        cache_subdir='datasets',
                        extract=True)

esc50_csv = './datasets/ESC-50-master/meta/esc50.csv'
base_data_path = './datasets/ESC-50-master/audio/'

In [ ]:
df = pd.read_csv(esc50_csv)
df.tail()

In [ ]:
df['category'].unique()

In [ ]:
classes = ['dog', 'door_wood_creaks', 'glass_breaking']
mapeamento = {'dog':0, 'door_wood_creaks':1,'glass_breaking':2}

In [ ]:
classes

In [ ]:
df_filtrado = df[df['category'].isin(classes)]

In [ ]:
df_filtrado

In [ ]:
df_filtrado.loc[:, 'alvo'] = df_filtrado['category'].apply(lambda name: mapeamento[name])

In [ ]:
full_path = df_filtrado['filename'].apply(lambda row: os.path.join(base_data_path, row))
df_filtrado = df_filtrado.assign(filename=full_path)
df_filtrado

## Transfer Learning - Utilizando embeddings

In [ ]:
def process_path(file_path, label, fold):
    audio = load_and_process_audio(file_path)
    label = tf.cast(label, tf.int64)  # Certifique-se de que label seja int64
    fold = tf.cast(fold, tf.int64)    # Certifique-se de que fold seja int64
    return audio, label, fold

# Criar um dataset do TensorFlow
def paths_labels_folds_to_dataset(audio_paths, labels, folds):
    path_ds = tf.data.Dataset.from_tensor_slices(audio_paths)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    fold_ds = tf.data.Dataset.from_tensor_slices(folds)
    audio_label_fold_ds = tf.data.Dataset.zip((path_ds, label_ds, fold_ds))
    return audio_label_fold_ds.map(lambda path, label, fold: tf.py_function(
        func=process_path, inp=[path, label, fold], Tout=[tf.float32, tf.int64, tf.int64]), num_parallel_calls=tf.data.AUTOTUNE)


# Função para preparar o dataset para o treinamento
def prepare_for_training(ds, batch_size=32, shuffle_buffer_size=1000):
    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
    ds = ds.batch(batch_size)
    #ds = ds.repeat(repeat_count)  # Adicionando o repeat
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

# Supondo que 'df' é o seu DataFrame
audio_paths = df_filtrado['filename'].tolist()
labels = df_filtrado['alvo'].tolist()
folds = df['fold'].tolist()

In [ ]:

# Criar o conjunto de dados do TensorFlow a partir do DataFrame
complete_dataset = paths_labels_folds_to_dataset(audio_paths, labels, folds)

In [ ]:
# Função para extrair embeddings usando o modelo YAMNet
def extract_embedding(wav_data, label, fold):
    ''' run YAMNet to extract embedding from the wav data '''
    scores, embeddings, spectrogram = model(wav_data)
    num_embeddings = tf.shape(embeddings)[0]
    return (embeddings,
            tf.repeat(label, num_embeddings),
            tf.repeat(fold, num_embeddings))

# Extrair embeddings e desagrupar o dataset
complete_dataset = complete_dataset.map(extract_embedding).unbatch()

In [ ]:
# Separar os dados em treinamento e validação com base nos folds
train_dataset = complete_dataset.filter(lambda audio, label, fold: fold < 5)  # Exemplo: folds 0-4 para treinamento
val_dataset = complete_dataset.filter(lambda audio, label, fold: fold == 5)   # Exemplo: fold 5 para validação

train_dataset = prepare_for_training(train_dataset)
val_dataset = prepare_for_training(val_dataset)
     

## Novo modelo

In [ ]:
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# Função para carregar e processar o áudio com resampling usando SciPy
def load_and_process_audio(filename, max_length=16000):
    file_contents = tf.io.read_file(filename)
    wav, sample_rate = tf.audio.decode_wav(file_contents, desired_channels=1)
    wav = tf.squeeze(wav, axis=-1)
    
    # Função de resampling usando SciPy
    def scipy_resample(wav, sample_rate):
        if sample_rate != 16000:
            wav = resample(wav, int(16000 / sample_rate * len(wav)))
        return wav

    # Usar tf.py_function para envolver a operação de resampling
    wav = tf.py_function(scipy_resample, [wav, sample_rate], tf.float32)
    
    # Adicionar padding ou cortar os sinais de áudio
    audio_length = tf.shape(wav)[0]
    if audio_length > max_length:
        wav = wav[:max_length]
    else:
        pad_length = max_length - audio_length
        paddings = [[0, pad_length]]
        wav = tf.pad(wav, paddings, "CONSTANT")
    
    return tf.reshape(wav, [max_length])

# Função para processar o caminho do arquivo de áudio, sua label e o fold
def process_path(file_path, label, fold):
    audio = load_and_process_audio(file_path)
    label = tf.cast(label, tf.int64)  # Certifique-se de que label seja int64
    fold = tf.cast(fold, tf.int64)    # Certifique-se de que fold seja int64
    return audio, label, fold

# Criar um dataset do TensorFlow
def paths_labels_folds_to_dataset(audio_paths, labels, folds):
    path_ds = tf.data.Dataset.from_tensor_slices(audio_paths)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    fold_ds = tf.data.Dataset.from_tensor_slices(folds)
    audio_label_fold_ds = tf.data.Dataset.zip((path_ds, label_ds, fold_ds))
    return audio_label_fold_ds.map(lambda path, label, fold: tf.py_function(
        func=process_path, inp=[path, label, fold], Tout=[tf.float32, tf.int64, tf.int64]), num_parallel_calls=tf.data.AUTOTUNE)

# Função para preparar o dataset para o treinamento
def prepare_for_training(ds, batch_size=32, shuffle_buffer_size=1000):
    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

# Função para extrair embeddings usando o modelo YAMNet
def extract_embedding(wav_data, label, fold):
    ''' run YAMNet to extract embedding from the wav data '''
    scores, embeddings, spectrogram = yamnet_model(wav_data)
    num_embeddings = tf.shape(embeddings)[0]
    return (embeddings,
            tf.repeat(label, num_embeddings),
            tf.repeat(fold, num_embeddings))

# Supondo que 'df' é o seu DataFrame
audio_paths = df_filtrado['filename'].tolist()
labels = df_filtrado['alvo'].tolist()
folds = df_filtrado['fold'].tolist()

# Criar o conjunto de dados do TensorFlow a partir do DataFrame
complete_dataset = paths_labels_folds_to_dataset(audio_paths, labels, folds)

# Verificar o dataset
for audio, label, fold in complete_dataset.take(1):
    print(f"Audio shape: {audio.shape}")
    print(f"Label: {label}")
    print(f"Fold: {fold}")

# Extrair embeddings e desagrupar o dataset
complete_dataset = complete_dataset.map(extract_embedding).unbatch()

# Verificar o dataset após extração de embeddings
for embeddings, label, fold in complete_dataset.take(1):
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Label: {label}")
    print(f"Fold: {fold}")

# Separar os dados em treinamento e validação com base nos folds
train_dataset = complete_dataset.filter(lambda embeddings, label, fold: fold < 5)  # Exemplo: folds 1-4 para treinamento
val_dataset = complete_dataset.filter(lambda embeddings, label, fold: fold == 5)   # Exemplo: fold 5 para validação

train_dataset = prepare_for_training(train_dataset)
val_dataset = prepare_for_training(val_dataset)

print(train_dataset.element_spec)
print(val_dataset.element_spec)

# Definir e compilar o modelo
meu_modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,), dtype=tf.float32, name='input_embedding'),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(len(np.unique(labels)))  # Certifique-se de que len(classes) está correto
])

meu_modelo.summary()

meu_modelo.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 optimizer="adam",
                 metrics=['accuracy'])

callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

# Treinar o modelo
history = meu_modelo.fit(train_dataset,
                       epochs=20,
                       validation_data=val_dataset,
                       callbacks=[callback])  # Certifique-se de que o callback é passado como uma lista


In [ ]:
def process_and_extract_embeddings(waveform, sample_rate, max_length=16000):
    # Função de resampling usando SciPy
    def scipy_resample(wav, sample_rate):
        if sample_rate != 16000:
            wav = resample(wav, int(16000 / sample_rate * len(wav)))
        return wav

    # Usar tf.py_function para envolver a operação de resampling
    wav = tf.py_function(scipy_resample, [waveform, sample_rate], tf.float32)
    
    # Adicionar padding ou cortar os sinais de áudio
    audio_length = tf.shape(wav)[0]
    if audio_length > max_length:
        wav = wav[:max_length]
    else:
        pad_length = max_length - audio_length
        paddings = [[0, pad_length]]
        wav = tf.pad(wav, paddings, "CONSTANT")
    
    wav = tf.reshape(wav, [max_length])

    # Extrair embeddings usando YAMNet
    scores, embeddings, spectrogram = yamnet_model(wav)
    return embeddings
     

def predict_audio_class(file_path, model, class_map):
    # Carregar e processar o arquivo de áudio MP3
    waveform, sample_rate = load_audio(file_path)
    
    # Processar e extrair embeddings
    embeddings = process_and_extract_embeddings(waveform, sample_rate)
    
    # Fazer a inferência com o modelo treinado
    predictions = model.predict(embeddings)
    
    # Agregar as previsões (por exemplo, usando a média das previsões)
    final_prediction = np.mean(predictions, axis=0)
    predicted_class_index = np.argmax(final_prediction)
    
    # Mapear o índice previsto para o nome da classe
    predicted_class_name = class_map[predicted_class_index]
    
    return predicted_class_name
     

mapeamento = {'dog': 0, 'door_wood_creaks': 1, 'glass_breaking': 2}
mapeamento_inverso = {v: k for k, v in mapeamento.items()}
     

# Exemplo de uso
file_path = '/teamspace/studios/this_studio/glass-hit-192119.mp3'
predicted_class = predict_audio_class(file_path, meu_modelo, mapeamento_inverso)

In [ ]:
predicted_class